<font color='blue'> **01 ________________________________________________________ GRID ________________________** </font> 

In [ ]:
import numpy as np

def Grid(lx, ly, lz, ncv):                            
    _ncv = ncv                                             # Store the number of control volumes
    dx = lx/float(ncv)                                     # Calculate the control volume length
    _xf = np.array([i*dx for i in range(ncv+1)])           # Calculate the face locations
    _xP = np.array([_xf[0]]       + [0.5*(_xf[i]+_xf[i+1]) for i in range(ncv)]       + [_xf[-1]])             # Calculate the cell centroid locations
    
    _Af = ly*lz*np.ones(ncv+1)                             # Calculate face areas
    _Ao = (2.0*dx*ly + 2.0*dx*lz)*np.ones(ncv)             # Calculate the outer surface area for each cell
    _vol = dx*ly*lz*np.ones(ncv)                           # Calculate cell volumes

def ncv(self):      return self._ncv     
def xf(self):       return self._xf
def xP(self):       return self._xP 
def dx_WP(self):    return self.xP[1:-1]-self.xP[0:-2]
def dx_PE(self):    return self.xP[2:]-self.xP[1:-1]
def Af(self):       return self._Af
def Aw(self):       return self._Af[0:-1]
def Ae(self):       return self._Af[1:]
def Ao(self):       return self._Ao
def vol(self):      return self._vol

In [ ]:
#  phi  # gamma
if   self._loc is BoundaryLocation.WEST:        self._phi[0] = self._value
elif self._loc is BoundaryLocation.EAST:        self._phi[-1] = self._value

flux_w = - self._gamma*self._grid.Aw * (self._phi[1:-1]-self._phi[0:-2]) /self._grid.dx_WP
flux_e = - self._gamma*self._grid.Ae * (self._phi[2:]  -self._phi[1:-1]) /self._grid.dx_PE

<font color='blue'> **02 ________________________________________________________ DiffusionModel ________________________** </font> 

In [ ]:
class DiffusionModel:
    def __init__(self, grid, phi, gamma, west_bc, east_bc):
        self._grid = grid   ; self._phi = phi   ; self._gamma = gamma   ; self._west_bc = west_bc   ; self._east_bc = east_bc

    def add(self, coeffs):

        # Calculate the west and east face diffusion flux terms for each face

        flux_w = - self._gamma*self._grid.Aw * (self._phi[1:-1]-self._phi[0:-2]) /self._grid.dx_WP  ; 
        flux_e = - self._gamma*self._grid.Ae * (self._phi[2:]  -self._phi[1:-1]) /self._grid.dx_PE

        # Calculate the linearization coefficients
        coeffW = - self._gamma*self._grid.Aw / self._grid.dx_WP     ;         coeffE = - self._gamma*self._grid.Ae / self._grid.dx_PE         ; coeffP = - coeffW - coeffE

        # Modify the linearization coefficients on the boundaries
        coeffP[0]  += coeffW[0]  * self._west_bc.coeff()    ; coeffP[-1] += coeffE[-1] * self._east_bc.coeff()

        # Zero the boundary coefficients that are not used
        coeffW[0] = 0.0 ; coeffE[-1] = 0.0

        # Calculate the net flux from each cell
        flux = flux_e - flux_w

        # Add to coefficient arrays
        coeffs.accumulate_aP(coeffP) ; coeffs.accumulate_aW(coeffW) ; coeffs.accumulate_aE(coeffE)  ; coeffs.accumulate_rP(flux)

        return coeffs